## Import Library

In [ ]:
from machine_lib import * 

## 1, Login
在machine_lib文件的login方法中填写用户名和密码后保存然后来到本文件Restart Kernal后重新import machine_lib

In [ ]:
s = login()

## 2, get data fields

In [ ]:
df = get_datafields(s, dataset_id = 'model26', region='USA', universe='TOP3000', delay=1)
print(df)

### Matrix Data

In [ ]:
print(df[df['type'] == "MATRIX"]["id"].tolist())
pc_fields = process_datafields(df, "matrix")
print(pc_fields)

### Vector Data

In [ ]:
df = get_datafields(s, dataset_id='news85', region='USA', universe='TOP3000', delay=1)
print(df[df['type'] == "VECTOR"]["id"].tolist())
print(process_datafields(df, "vector"))

## 3, Alpha factory
### start with First Order

In [ ]:
first_order = first_order_factory(pc_fields, ts_ops)
print(first_order[:10])
print(len(first_order))

In [ ]:
# Pad initial decay with alpha
init_decay = 6
fo_alpha_list = []
for alpha in first_order:
    fo_alpha_list.append((alpha, init_decay))
random.shuffle(fo_alpha_list)
print(len(fo_alpha_list))
print(fo_alpha_list[:5])

In [ ]:
# Load alphas to task pools
fo_pools = load_task_pool(fo_alpha_list, 10, 9)
print(fo_pools[0])

## 4, simulate alphas

In [ ]:
# Simulate First Order
multi_simulate(fo_pools, "SUBINDUSTRY", "USA", "TOP3000", 0)

## 5, Select alphas
go to web alphas penal to look for the number and date to track for next order improve

In [ ]:
## get promising alphas to improve in the next order
fo_tracker = get_alphas("01-22", "01-23", 1.2, 1, "USA", 100, "track")
print(len(fo_tracker))

#### Prune 剪枝

In [ ]:
fo_layer = prune(fo_tracker, 'mdl26', 5)

## 6, Next order improvement - Second Order
second order: ts_ops(field, days) -> group_ops(ts_ops(field, days), group)

In [ ]:
so_alpha_list = []
group_ops = ["group_neutralize", "group_rank", "group_zscore"]

for expr, decay in fo_layer:
    for alpha in get_group_second_order_factory([expr], group_ops, "USA"):
        so_alpha_list.append((alpha,decay))

random.shuffle(so_alpha_list)
print(len(so_alpha_list))
print(so_alpha_list[:3])

### Simulate second order

In [ ]:
so_pools = load_task_pool(so_alpha_list, 10, 9)
multi_simulate(so_pools, 'SUBINDUSTRY', 'USA', 'TOP3000', 0)

## Higher Order for improvement - Third Order
group_ops(ts_ops(field, days), group) -> trade_when(entre_event, group_ops(ts_ops(field, days), group), exit_event)

In [ ]:
## get promising alphas from second order to improve in the third order
so_tracker = get_alphas("01-22", "01-23", 1.4, 1, "USA", 110, "track")

print(len(so_tracker))

so_layer = prune(so_tracker, 'mdl26', 5)
th_alpha_list = []

for expr, decay in so_layer:
    for alpha in trade_when_factory("trade_when",expr,"USA"):
        th_alpha_list.append((alpha,decay))

random.shuffle(th_alpha_list)        
print(len(th_alpha_list))

### Simulate Third Order

In [ ]:
# Simulate third order
th_pools = load_task_pool(th_alpha_list, 10, 9)
multi_simulate(th_pools, 'SUBINDUSTRY', 'USA', 'TOP3000', 0)

## 7, Get submittable alphas


In [ ]:
# 1.58 sharpe, 1 fitness, "submit"参数
th_tracker = get_alphas("07-16", "07-20", 1.58, 1, "USA", 200, "submit")

In [ ]:
## 将get的alpha的id取出至stone_bag，用api check submission
stone_bag = []
for alpha in th_tracker:
    stone_bag.append(alpha[0])
print(len(stone_bag))
gold_bag = []
check_submission(stone_bag, gold_bag, 0)

In [ ]:
# 打印可提交的alpha信息并按sharpe排序，在网页上找到alpha手动提交
view_alphas(gold_bag)

## 8, fine-tune submittable alphas
neutralization, performance comparison, turnover, margin